# Inpainting submarino -> tokens DINOv3-VQ (Kaggle)
Requisitos: GPU (T4/P100), Internet ON, secret `HF_TOKEN` (acceso aprobado a `facebook/dinov3-vitb16-pretrain-lvd1689m`),
y un Dataset con: `common.py train.py predict.py dino_vq.py codebook.npy target.csv train.zip test.zip` (o carpetas `train/` `test/`).
Ejecutar con **Save & Run All** (commit en segundo plano).

In [1]:
import os, glob, subprocess, shutil, sys
# --- Config ---
SMOKE = False            # True: prueba rápida (60 pasos) antes del run largo
TIME_BUDGET_H = 9.5      # el commit de Kaggle dura máx ~12h; deja margen para predict
RESUME_FROM = ""         # p.ej. /kaggle/input/<ckpt-dataset>/last.pt para continuar un entrenamiento
# --- Localizar dataset ---
hits = glob.glob('/kaggle/input/datasets/pieropr2000/aquatic/target.csv', recursive=True)
#hits = glob.glob('/kaggle/input/**/target.csv', recursive=True)
assert hits, 'No encuentro target.csv en /kaggle/input'
SRC = os.path.dirname(hits[0]); print('SRC =', SRC)
W = '/kaggle/working'; T = '/kaggle/temp'; os.makedirs(T, exist_ok=True)
for f in ['common.py','train.py','predict.py','dino_vq.py']:
    shutil.copy(glob.glob(f'/kaggle/input/**/{f}', recursive=True)[0], W)
shutil.copy(f'{SRC}/codebook.npy', W); shutil.copy(f'{SRC}/target.csv', W)
os.chdir(W)

SRC = /kaggle/input/datasets/pieropr2000/aquatic


In [2]:
def find_dir(name):
    d = f'{SRC}/{name}'
    if os.path.isdir(d): return d
    z = f'{SRC}/{name}.zip'
    if os.path.exists(z):
        subprocess.run(['unzip','-q','-o',z,'-d',f'{T}/{name}_x'], check=True)
        sub = f'{T}/{name}_x/{name}'
        return sub if os.path.isdir(sub) else f'{T}/{name}_x'
    raise FileNotFoundError(name)
TRAIN, TEST = find_dir('train'), find_dir('test')
print(TRAIN, len(os.listdir(TRAIN)), '|', TEST, len(os.listdir(TEST)))

/kaggle/input/datasets/pieropr2000/aquatic/train 1 | /kaggle/input/datasets/pieropr2000/aquatic/test 1


In [3]:
!pip install -q -U transformers
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = secret_value_0
os.environ["HUGGING_FACE_HUB_TOKEN"] = secret_value_0
login(secret_value_0)

import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2.11.0+cu130 True Tesla T4


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = f"python -u train.py --train-dir {TRAIN} --codebook codebook.npy --out runs/v1 --cache {T}/train.npy --bs 12"
if SMOKE: cmd += " --max-steps 60 --val-n 50 --val-every 30 --time-budget-h 0.2"
else:     cmd += f" --time-budget-h {TIME_BUDGET_H}"
if RESUME_FROM: cmd += f" --resume {RESUME_FROM}"
!{cmd}

15252 imágenes de train
train=14752 val=500
config.json: 100%|█████████████████████████████| 744/744 [00:00<00:00, 3.11MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes: ████▍                 | 70.0MB, 2.96MB/s  
model.safetensors: downloading bytes: ██████▏               | 97.2MB, 6.34MB/s  
model.safetensors: reconstructing file:  29%|█▋    | 99.0MB /  343MB, 7.08MB/s  
model.safetensors: downloading bytes: █████████████▋        |  214MB, 15.9MB/s  
model.safetensors: downloading bytes: ████████████████████▋ |  321MB, 25.6MB/s  
model.safetensors: downloading bytes: ██████████████████████|  325MB, 25.9MB/s  
model.safetensors: reconstructing file: 100%|██████|  343MB /  343MB, 28.5MB/s  
Loading weights: 100%|██████████████████████| 211/211 [00:00<00:00, 2602.29it/s]
parámetros U-Net: 34.7M
baseline parche negro: VAL acc=0.0198
step 200 t=0.05h lr=1.20e-04 loss=4.0340 pix=0.1101 cos=0.4332 ce=6.115
step 400 t=0.10h lr=2.40e-0

In [5]:
!python predict.py --ckpt runs/v1/last.pt --test {TEST} --target target.csv --out generate --tta

  0%|                                                  | 0/1056 [00:00<?, ?it/s]/kaggle/working/predict.py:31: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(orig).to(dev).permute(2, 0, 1)[None].float() / 255
100%|███████████████████████████████████████| 1056/1056 [02:34<00:00,  6.83it/s]
listo -> generate


In [6]:
!python dino_vq.py --images-root generate --target-csv target.csv --codebook-path codebook.npy --output-csv submission.csv
import pandas as pd
s = pd.read_csv('submission.csv'); print(s.shape, s.Target.min(), s.Target.max(), s.Id.duplicated().sum())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Generando submission: 100%|███████████████████| 132/132 [00:07<00:00, 16.66it/s]
Archivo generado: submission.csv
Filas: 14913
(14913, 2) 0 1023 0


In [7]:
# Opcional: liberar espacio de salida (solo quedan last.pt y submission.csv)
shutil.rmtree('generate', ignore_errors=True)